# 原 Stage1 0bp：全部 819 个 126 日窗口的现金与 SOXL 选择归因

只读已验证原生 Qlib Recorder；静态分量修复只说明原账户路径上的必要缺口，不是交易反事实。冻结计划见 `plan.json`。

In [1]:
import os,sys,json,hashlib,math
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
card=root/'research/stage1_original_0bp_all_window_components_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();parent=root/src['component_card'];assert sha(parent/'summary.json')==src['component_summary_sha256']
summary=json.loads((parent/'summary.json').read_text());assert summary['recorder_id']==src['recorder_id'] and summary['experiment_id']==src['experiment_id'] and summary['nas_evidence']['sha256']==src['NAS_archive_sha256'] and summary['nas_readback']['verified']
a=parent/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts';assert sha(a/src['rolling_artifact'])==src['rolling_artifact_sha256'] and sha(a/src['daily_artifact'])==src['daily_artifact_sha256']
parent_plan=json.loads((parent/'plan.json').read_text());assert sha(parent/'plan.json')==summary['plan_sha256']
state={'plan_sha256':sha(card/'plan.json'),'phase':'readback_attempted','source_Recorder_object_load_attempts':2,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Qlib_Recorders':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/parent_plan['sources']['provider_uri']),region='us',kernels=1)
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']);assert rec.status=='FINISHED';windows=rec.load_object(src['rolling_artifact']);daily=rec.load_object(src['daily_artifact'])
assert len(windows)==819 and len(daily)==944 and windows.sessions.eq(126).all()
fields=['cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']
assert np.isfinite(windows[['relative_log_excess',*fields]].to_numpy(float)).all() and np.max(np.abs(windows[fields].sum(axis=1)-windows.relative_log_excess))<1e-10
assert np.max(np.abs(daily[fields].sum(axis=1)-daily.paired_log_excess))<1e-10
win=windows.copy();pos=win.relative_log_excess>0;neg=~pos;cashneg=win.cash_opportunity_log<0;selneg=win.SOXL_selection_log<0
quadrants={}
for group,mask in [('negative_or_zero',neg),('strict_positive',pos)]:
 quadrants[group]={'cash_negative_selection_negative':int((mask&cashneg&selneg).sum()),'cash_negative_selection_nonnegative':int((mask&cashneg&~selneg).sum()),'cash_nonnegative_selection_negative':int((mask&~cashneg&selneg).sum()),'cash_nonnegative_selection_nonnegative':int((mask&~cashneg&~selneg).sum())}
def descriptive(mask):
 z=win.loc[mask]
 return {'windows':len(z),'median_relative_log_excess':float(z.relative_log_excess.median()),'mean_cash_opportunity_log':float(z.cash_opportunity_log.mean()),'mean_SOXL_selection_log':float(z.SOXL_selection_log.mean()),'mean_fee_log':float(z.Stage1_direct_fees_log.mean()),'mean_timing_residual_log':float(z.timing_and_benchmark_residual_log.mean()),'mean_cap_binding_fraction':float(z.cap_binding_fraction.mean()),'mean_ending_beta':float(z.mean_ending_beta.mean())}
repaired_cash=win.relative_log_excess-win.cash_opportunity_log.clip(upper=0);repaired_selection=win.relative_log_excess-win.SOXL_selection_log.clip(upper=0);repaired_both=repaired_cash-win.SOXL_selection_log.clip(upper=0)
need=math.ceil(.7*len(win));ranked=np.sort(win.relative_log_excess.to_numpy(float))[::-1]
result={'decision':'descriptive_all_819_Stage1_0bp_window_component_gap_no_policy','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'full_sessions':len(daily),'rolling126_windows':len(win),'rolling70pct_required_strict_positive_windows':need,'original_strict_positive_windows':int(pos.sum()),'original_positive_fraction':float(pos.mean()),'additional_positive_windows_needed':int(max(0,need-int(pos.sum()))),'negative_window_component_quadrants':quadrants['negative_or_zero'],'positive_window_component_quadrants':quadrants['strict_positive'],'descriptive':{'all':descriptive(np.ones(len(win),dtype=bool)),'negative_or_zero':descriptive(neg),'strict_positive':descriptive(pos)},'static_direct_component_repair_positive_windows':{'cash_negative_to_zero_only':int((repaired_cash>0).sum()),'selection_negative_to_zero_only':int((repaired_selection>0).sum()),'both_negative_to_zero':int((repaired_both>0).sum())},'uniform_126_window_log_lift_infimum_for_70pct':float(max(0,-ranked[need-1])),'windows_overlap':True,'static_repairs_are_native_accounts':False,'causal_policy_inference':False,'market_requests':0,'model_fits':0,'native_accounts':0,'bootstraps':0}
assert sum(result['negative_window_component_quadrants'].values())==int(neg.sum()) and sum(result['positive_window_component_quadrants'].values())==int(pos.sum())
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
small={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':src['recorder_id'],'source_NAS_sha256':src['NAS_archive_sha256'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['819 windows overlap heavily and are not independent trials.','Direct additive component repair freezes original account path; actual actions alter risk state, fees and subsequent positions.','Only retrospective attribution; no new valid factor or SOXX hard-gate pass.']};(card/'summary.json').write_text(json.dumps(small,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(result)

[38177:MainThread](2026-10-23 09:01:04,719) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[38177:MainThread](2026-10-23 09:01:04,721) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[38177:MainThread](2026-10-23 09:01:04,721) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'decision': 'descriptive_all_819_Stage1_0bp_window_component_gap_no_policy', 'plan_sha256': 'f52438ab827fd482a965d6c4a317d0cae11ff6c5e54c9837d9e68181d9ed0917', 'source_recorder_id': '2f0392615c4c4b15928b4ada6e01ecd9', 'full_sessions': 944, 'rolling126_windows': 819, 'rolling70pct_required_strict_positive_windows': 574, 'original_strict_positive_windows': 255, 'original_positive_fraction': 0.31135531135531136, 'additional_positive_windows_needed': 319, 'negative_window_component_quadrants': {'cash_negative_selection_negative': 214, 'cash_negative_selection_nonnegative': 322, 'cash_nonnegative_selection_negative': 28, 'cash_nonnegative_selection_nonnegative': 0}, 'positive_window_component_quadrants': {'cash_negative_selection_negative': 0, 'cash_negative_selection_nonnegative': 94, 'cash_nonnegative_selection_negative': 70, 'cash_nonnegative_selection_nonnegative': 91}, 'descriptive': {'all': {'windows': 819, 'median_relative_log_excess': -0.055307212484741256, 'mean_cash_opportunity_l

# 原 Stage1 0bp：全 126 日窗口现金与 SOXL 选择归因

只读已验证的完整 Stage1 自有状态、10 万美元、0bp 额外滑点且仍扣 IBKR Tiered 费用的原生 Qlib Recorder `2f0392615c4c4b15928b4ada6e01ecd9`；其 NAS 归档 SHA-256 `54c6e93b…` 已在上游限定恢复。本卡读取原生全 944 日及全部 819 个连续 126 日窗口，不新增市场请求、拟合、账户或 bootstrap。

|固定窗口统计|结果|
|---|---:|
|原账户严格胜 SOXX|255/819，31.14%|
|合同 70% 所需|574/819，尚缺319窗|
|未胜窗口中现金分量为负|536/564|
|未胜窗口中 SOXL 选择分量为负|242/564|
|现金与选择同时为负|214/564|
|现金负、选择非负|322/564|
|现金非负、选择负|28/564|

在原始账本上做**静态分量替换**：只把负 SOXL 选择分量改为零，可得282个胜窗；只把负现金机会分量改为零，可得576个胜窗，仅比574门槛多2个；两者都改为零为622个。要让排序第574的窗口严格转正，对**每个**窗口至少需要约0.10545的统一126日对数超额增量下确界。它们都不是可执行或因果反事实：一旦改持仓，风险状态、费用、后续资产和窗口结果都会变化，年度、回撤、CVaR和区块下界也没有因此过门。

**研究方向收敛到广泛的现金机会损失。** 只修2026年7月少数 SOXL 亏损日或仅改善负选择窗口，不能弥补所需319个胜窗。下一特征须在决策时识别“应保留现金保护”与“持续现金拖累”两类情形，同时在完整 Stage1 账户检验其他年份、季度及全部窗口；不能按本表的负分量、后验窗口或股价结果制定开关。819个窗口高度重叠，历史已反复使用，不是独立验证。
